In [1]:
import os, requests, textwrap, html, re
from typing import List, Dict
from dotenv import load_dotenv
import pprint

load_dotenv()

True

In [2]:
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("No 'OPENAI_API_KEY' found in the environment variables")

In [3]:
from rich.console import Console
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from rich.table import Table
from rich.prompt import Prompt, Confirm

console = Console()

console.print("[cyan bold]Hello Learner[/cyan bold]")

Hello Learner

# Main

In [25]:
def main() -> None:
    console.print("[bold blue]Welcome User[/bold blue]")

    while True:
        query = Prompt.ask("[bold]What do you want to learn?[/bold]").strip()

        if query.lower() == "exit":
            console.print("[bold yellow]Goodbye![/bold yellow]")
            break
    
        if not query:
            console.print("[bold red]Error: [/bold red]Please provide a valid topic.")
            continue
            
    
        console.print(f"[bold cyan]The user want's to learn about {query}[/bold cyan]")
        break

In [28]:
main()

Welcome User

What do you want to learn?:

 exit


Goodbye!

# Orchestrator

In [4]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Callable, List, Optional, Tuple
from pathlib import Path
import hashlib
import json
import re
from datetime import datetime

from pydantic import BaseModel, Field
from agents import Agent, Runner, RunConfig, ModelSettings
import nest_asyncio, asyncio

CONTENT_DIR = Path("content")
CONTENT_DIR.mkdir(parents=True, exist_ok=True)


APPROVE_ANYWAY = "__FORCE_APPROVE__"

In [5]:
class Module(BaseModel):
    name: str
    lessons: List[str] = Field(..., description="3–7 concise lesson titles in logical order")

In [6]:
class Curriculum(BaseModel):
    topic: str
    level: str = "beginner"
    goal: Optional[str] = None
    modules: List[Module]

In [7]:
class Review(BaseModel):
    approved: bool
    issues: List[str] = []
    revision_instructions: str = ""

In [8]:
class DetailedTopic(BaseModel):
    title: str
    subtopic: List[str] = Field(..., description="Bulleted, actionable learning points")

In [9]:
class DetailedSyllabus(BaseModel):
    topic: str
    outline: List[DetailedTopic]

In [13]:
@dataclass
class Orchestrator:
    model: str = "gpt-4o-mini"
    temperature: float = 0.2

    def __post_init__(self):
        self.curriculum_agent = Agent(
            name="Curriculum Planner",
            instructions=(
                "You create tight, pragmatic learning curricula. "
                "Given a topic, level, goal, and any user change requests, "
                "produce a short curriculum with 3–6 modules. Keep lesson titles concise. "
                "Prefer fundamentals first, then practice."
            ),
            output_type=Curriculum
        )

        self.reviewer_agent = Agent(
            name="Curriculum Reviewer",
            instructions=(
                "You critically review a curriculum for prerequisite order, scope creep, "
                "jargon, and uneven load. If fixes are needed, set approved=false and write "
                "a single, crisp set of revision instructions."
            ),
            output_type=Review,
        )

        self.detail_agent = Agent(
            name="Detail Drafter",
            instructions=(
                "Expand an approved curriculum into a detailed syllabus. "
                "For each module, produce 4–8 actionable subtopics (bullets). Be specific; "
                "avoid fluff (no 'learn basics')."
            ),
            output_type=DetailedSyllabus,
        )

        self._run_config = RunConfig(
            model=self.model,
            model_settings=ModelSettings(temperature=self.temperature),
            workflow_name="LearningPlatform_MVP",
        )

    async def plan_curriculum(
        self,
        topic: str,
        level: str = "beginner",
        goal: Optional[str] = None,
        change_request: Optional[str] = None,
    ) -> Curriculum:
        prompt = (
            f"Topic: {topic}\nLevel: {level}\nGoal: {goal or 'Not specified'}\n"
            f"Change request (if any): {change_request or 'None'}\n"
            "Return a Curriculum object."
        )
        res = await Runner.run(self.curriculum_agent, prompt, run_config=self._run_config)
        return res.final_output


    async def review(self, curriculum: Curriculum) -> Review:
        prompt = (
            "Review the following curriculum. Approve only if it is clear, ordered, "
            "balanced, and free of jargon.\n\n"
            f"{curriculum.model_dump_json(indent=2)}\n"
            "Return a Review object."
        )
        console.print(Panel(f"[bold cyan]Review Prompt: [/bold cyan] \n[dim] {prompt} [/dim]")) # check review prompt
        res = await Runner.run(self.reviewer_agent, prompt, run_config=self._run_config)
        return res.final_output

    # revise loop
    async def revise_until_approved(
        self,
        initial: Curriculum,
        get_user_feedback: Optional[Callable[[Curriculum, Review], Optional[str]]] = None,
        max_loops: int = 1,
    ) -> Curriculum:
        """Deterministic loop: (plan -> review -> optional user change -> re-plan)"""
        curriculum = initial
        for _ in range(max_loops):
            review = await self.review(curriculum)

            if review.approved:
                return curriculum

            # Merge reviewer instructions + (optional) user feedback into a single, crisp change request.
            user_change = get_user_feedback(curriculum, review) if get_user_feedback else None
            merged_change = self._merge_changes(review.revision_instructions, user_change)

            curriculum = await self.plan_curriculum(
                topic=curriculum.topic,
                level=curriculum.level,
                goal=curriculum.goal,
                change_request=merged_change,
            )

        # Last resort: return latest even if not approved
        return curriculum


    async def draft_details(self, curriculum: Curriculum) -> DetailedSyllabus:
        prompt = (
            "Expand this approved curriculum into a detailed syllabus with topics and bulleted subtopics.\n\n"
            f"{curriculum.model_dump_json(indent=2)}\n"
            "Return a DetailedSyllabus."
        )
        res = await Runner.run(self.detail_agent, prompt, run_config=self._run_config)
        return res.final_output  # -> DetailedSyllabus


    # ---- Public API ----
    async def orchestrate(
        self,
        topic: str,
        level: str = "beginner",
        goal: Optional[str] = None,
        get_user_feedback: Optional[Callable[[Curriculum, Review], Optional[str]]] = None,
    ) -> DetailedSyllabus:
        draft = await self.plan_curriculum(topic, level, goal)
        approved = await self.revise_until_approved(draft, get_user_feedback=get_user_feedback)
        return await self.draft_details(approved)


    # ---- Utils ----

    @staticmethod
    def _merge_changes(reviewer_instr: str, user_change: Optional[str]) -> str:
        parts = []
        if reviewer_instr and reviewer_instr.strip():
            parts.append(f"Reviewer: {reviewer_instr.strip()}")
        if user_change and user_change.strip():
            parts.append(f"User: {user_change.strip()}")
        return "\n".join(parts) if parts else "No changes—tighten clarity and ordering."


In [16]:
from typing import Optional

def _cli_feedback(curriculum: Curriculum, review: Review) -> Optional[str]:
    print("\n--- REVIEW ---")
    print(f"Approved: {review.approved}")
    if review.issues:
        print("Issues:")
        for i, iss in enumerate(review.issues, 1):
            print(f"  {i}. {iss}")
    print(f"Proposed revision instructions: {review.revision_instructions}\n")

    ans = input("Accept reviewer’s changes? (y=accept & continue, n=add your own, a=approve anyway): ").strip().lower()
    if ans == "a":
        return None
    if ans == "y":
        return review.revision_instructions
    if ans == "n":
        user = input("Describe your change request in one or two sentences: ").strip()
        return user
    return review.revision_instructions

print("=== Learning Platform Orchestrator (MVP) ===")
topic = input("What do you want to learn? ").strip()
level = input("Level (beginner/intermediate/advanced) [beginner]: ").strip() or "beginner"
goal = input("Goal (optional): ").strip() or None

orch = Orchestrator()  # methods use await Runner.run(...)

draft = await orch.plan_curriculum(topic=topic, level=level, goal=goal)
print("\n--- INITIAL CURRICULUM DRAFT ---")
print(draft.model_dump_json(indent=2))

approved = await orch.revise_until_approved(draft, get_user_feedback=_cli_feedback)
print("\n--- APPROVED CURRICULUM ---")
print(approved.model_dump_json(indent=2))

detailed = await orch.draft_details(approved)
print("\n--- DETAILED SYLLABUS ---")
console.print(Panel(detailed.model_dump_json(indent=1)))


=== Learning Platform Orchestrator (MVP) ===


What do you want to learn?  history of human kind
Level (beginner/intermediate/advanced) [beginner]:  
Goal (optional):  



--- INITIAL CURRICULUM DRAFT ---
{
  "topic": "history of human kind",
  "level": "beginner",
  "goal": null,
  "modules": [
    {
      "name": "Introduction to Human History",
      "lessons": [
        "What is History?",
        "Prehistoric Times",
        "The Rise of Civilizations"
      ]
    },
    {
      "name": "Ancient Civilizations",
      "lessons": [
        "Mesopotamia and Egypt",
        "Indus Valley and China",
        "Greece and Rome"
      ]
    },
    {
      "name": "Middle Ages to Renaissance",
      "lessons": [
        "The Fall of Rome",
        "Feudalism and the Middle Ages",
        "The Renaissance and Exploration"
      ]
    },
    {
      "name": "Modern History",
      "lessons": [
        "The Age of Enlightenment",
        "Industrial Revolution",
        "World Wars"
      ]
    },
    {
      "name": "Contemporary Issues",
      "lessons": [
        "Post-War Era",
        "Globalization",
        "Current Trends in History"
      ]
    }
  ]


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Review Prompt:                                                                                                  │
│  Review the following curriculum. Approve only if it is clear, ordered, balanced, and free of jargon.           │
│                                                                                                                 │
│ {                                                                                                               │
│   "topic": "history of human kind",                                                                             │
│   "level": "beginner",                                                                                          │
│   "goal": null,                                                                                                 │
│   "modules": [                                                                                                  │
│     {                                                                                                           │
│       "name": "Introduction to Human History",                                                                  │
│       "lessons": [                                                                                              │
│         "What is History?",                                                                                     │
│         "Prehistoric Times",                                                                                    │
│         "The Rise of Civilizations"                                                                             │
│       ]                                                                                                         │
│     },                                                                                                          │
│     {                                                                                                           │
│       "name": "Ancient Civilizations",                                                                          │
│       "lessons": [                                                                                              │
│         "Mesopotamia and Egypt",                                                                                │
│         "Indus Valley and China",                                                                               │
│         "Greece and Rome"                                                                                       │
│       ]                                                                                                         │
│     },                                                                                                          │
│     {                                                                                                           │
│       "name": "Middle Ages to Renaissance",                                                                     │
│       "lessons": [                                                                                              │
│         "The Fall of Rome",                                                                                     │
│         "Feudalism and the Middle Ages",                                                                        │
│         "The Renaissance and Exploration"                                                                       │
│       ]                                                                                                         │
│     },                                                                                                          │
│     {                                                                                                           │
│       "name": "Modern History",                       


--- REVIEW ---
Approved: False
Issues:
  1. Lack of clear learning goals for each module
  2. Some modules may have overlapping content leading to scope creep
  3. Jargon such as 'Globalization' and 'Current Trends in History' may not be clear to beginners
  4. Uneven load in modules, with some covering broad periods while others are more focused
Proposed revision instructions: Define clear learning objectives for each module, ensure content is distinct to avoid overlap, simplify or explain jargon, and balance the scope of each module to provide a more uniform workload.



Accept reviewer’s changes? (y=accept & continue, n=add your own, a=approve anyway):  



--- APPROVED CURRICULUM ---
{
  "topic": "history of human kind",
  "level": "beginner",
  "goal": null,
  "modules": [
    {
      "name": "Introduction to Human History",
      "lessons": [
        "What is Human History?",
        "Key Concepts and Terms",
        "Overview of Prehistoric Times"
      ]
    },
    {
      "name": "Early Civilizations",
      "lessons": [
        "The Rise of Agriculture",
        "Mesopotamia: The Cradle of Civilization",
        "Ancient Egypt: Culture and Society"
      ]
    },
    {
      "name": "Classical Civilizations",
      "lessons": [
        "Greece: Democracy and Philosophy",
        "Rome: Empire and Governance",
        "China: Dynasties and Innovations"
      ]
    },
    {
      "name": "Medieval to Modern Times",
      "lessons": [
        "The Middle Ages: Feudalism and Culture",
        "The Renaissance: Rebirth of Ideas",
        "The Age of Exploration: New Worlds"
      ]
    },
    {
      "name": "Recent History",
      "le

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│  "topic": "history of human kind",                                                                              │
│  "outline": [                                                                                                   │
│   {                                                                                                             │
│    "title": "Introduction to Human History",                                                                    │
│    "subtopic": [                                                                                                │
│     "Define human history and its significance in understanding humanity.",                                     │
│     "Identify key concepts such as chronology, historiography, and primary vs. secondary sources.",             │
│     "Explore the timeline of prehistoric times, including major milestones like the Stone Age and the           │
│ development of language.",                                                                                      │
│     "Discuss the importance of archaeological findings in reconstructing human history."                        │
│    ]                                                                                                            │
│   },                                                                                                            │
│   {                                                                                                             │
│    "title": "Early Civilizations",                                                                              │
│    "subtopic": [                                                                                                │
│     "Analyze the transition from nomadic lifestyles to settled agricultural societies.",                        │
│     "Examine the geographical and environmental factors that contributed to the rise of Mesopotamia.",          │
│     "Investigate the social, political, and economic structures of Ancient Egypt.",                             │
│     "Explore the contributions of early civilizations to writing, architecture, and governance."                │
│    ]                                                                                                            │
│   },                                                                                                            │
│   {                                                                                                             │
│    "title": "Classical Civilizations",                                                                          │
│    "subtopic": [                                                                                                │
│     "Discuss the development of democracy in Ancient Greece and its lasting impact on modern governance.",      │
│     "Examine the structure of the Roman Empire and its influence on law, culture, and infrastructure.",         │
│     "Analyze the contributions of Chinese dynasties to science, technology, and philosophy.",                   │
│     "Explore the interactions between these civilizations through trade, war, and cultural exchange."           │
│    ]                                                                                                            │
│   },                                                                                                            │
│   {                                                                                                             │
│    "title": "Medieval to Modern Times",                                                                         │
│    "subtopic": [                                      